# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing all schema entities by their `@id`.

### Dataset Source
This dataset is defined by the Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and instantiate the Croissant Dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

dataset = mlc.Dataset(url)

# Access metadata attributes via attribute access
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print(f"Published: {meta.datePublished}\n")
print(f"Version: {meta.version}\n")
print(f"Identifier: {meta.identifier}\n")
print(f"License: {meta.license}\n")

## 2. Data Overview
Review available **record sets** and their referenced fields and columns, each by `@id`.

> The major dataset entities are referenced by their schema `@id`. Let's print the available record sets, their `@id`, and the respective fields within each.

In [ ]:
from pprint import pprint

# List all RecordSets by @id
record_sets = meta.recordSet if hasattr(meta, 'recordSet') else []
if record_sets:
    print('Available Record Sets:')
    for rs in record_sets:
        print(f"  RecordSet @id: {rs['@id']}")
        # Try to list associated fields (by @id)
        if 'field' in rs:
            fields = rs['field']
            if not isinstance(fields, list):
                fields = [fields]
            print("    Fields:")
            for field in fields:
                print(f"      - {field['@id']} (name={field['name']})")
        else:
            print("    No fields listed.")
else:
    print('No record sets referenced in top-level metadata. Attempting to inspect dataset.records...\n')
    # Try loading one record from the default record set to infer structure
    records_preview = list(dataset.records(limit=1))
    if records_preview:
        print('Sample Record:')
        pprint(records_preview[0])
    else:
        print('No records available.')

## 3. Data Extraction
Extract data by **record set** using the proper `@id`. If the dataset defines explicit record sets, use them; otherwise, use the default record set, and reference fields by `@id`.

We'll load the principal data table into a DataFrame. If only one main record set exists, its `@id` will be used; otherwise, specify the record set(s) you wish to extract.

In [ ]:
# Attempt to enumerate all Record Set @ids (if available); else default to main dataset
record_set_ids = []
if hasattr(meta, 'recordSet') and meta.recordSet:
    for rs in meta.recordSet:
        record_set_ids.append(rs['@id'])
else:
    # Try loading records without specifying a record set (default set)
    # This dataset may not define record sets explicitly, as sometimes occurs with simple tables.
    record_set_ids = [None]

dataframes = {}

# Load DataFrame(s) by record set
for rsid in record_set_ids:
    if rsid is not None:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
    else:
        records = list(dataset.records())
        df = pd.DataFrame(records)
        dataframes['main'] = df

# Preview DataFrame
df_key = record_set_ids[0] if record_set_ids and record_set_ids[0] is not None else 'main'
print('Columns in main DataFrame:', dataframes[df_key].columns.tolist())
dataframes[df_key].head()

## 4. Exploratory Data Analysis (EDA)
Process and explore the main dataset by filtering, normalizing, and grouping over key fields (referenced via `@id`).

> For illustration, we'll select a numeric field (e.g., `MW` for molecular weight, or other quantitative protein attribute, using its schema `@id` if available) and run some basic operations, while referencing the raw DataFrame columns. Make sure to select the actual field names present in the DataFrame from above.

In [ ]:
# Identify numeric fields (print all columns for help)
print('Columns:', dataframes[df_key].columns.tolist())

# Select a quantitative field by column name (use actual field name from previous output; fallback example 'MW')
numeric_field = None
for candidate in ['MW', 'molecular_weight', 'Molecular Weight', 'molecular_weight_Da', 'coverage_percent', 'normalized_abundance']:
    if candidate in dataframes[df_key].columns:
        numeric_field = candidate
        break
if numeric_field is None:
    raise ValueError("No numeric field found. Please set a field present in the DataFrame.")

# Ensure the field is numeric type
dataframes[df_key][numeric_field] = pd.to_numeric(dataframes[df_key][numeric_field], errors='coerce')

threshold = dataframes[df_key][numeric_field].quantile(0.9)  # Filter top 10% as example
filtered_df = dataframes[df_key][dataframes[df_key][numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df[[numeric_field]].head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field
group_candidates = ['description', 'Description', 'accession', 'sample_id']
group_field = None
for candidate in group_candidates:
    if candidate in dataframes[df_key].columns:
        group_field = candidate
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped data by '{group_field}':")
    print(grouped_df.head())
else:
    print("\nNo group field found for aggregation.")

## 5. Visualization
Visualize the distribution of a numeric attribute (e.g., molecular weight) and relationship to a group/category if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for numeric field
plt.figure(figsize=(8,4))
sns.histplot(dataframes[df_key][numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field is available, boxplot
if group_field:
    plt.figure(figsize=(10,5))
    # Only plot top groups (by count)
    top_groups = dataframes[df_key][group_field].value_counts().index[:5]
    sns.boxplot(x=group_field, y=numeric_field, data=dataframes[df_key][dataframes[df_key][group_field].isin(top_groups)])
    plt.title(f"{numeric_field} by {group_field} (top 5 groups)")
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading, exploring, and processing the [FAIR^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) via its Croissant schema using the `mlcroissant` Python library. All schema elements (record sets, fields) were referenced by their `@id` as per the Croissant specification. 

**Key findings:**
- Dataset columns were explored and visualized.
- Numeric fields (e.g., molecular weights) can be filtered and normalized for downstream analysis.
- Grouping and summary statistics support biological insights into protein features.

Further analyses may include domain-specific modelling, advanced data integration, or machine learning approaches using the processed DataFrames.